In [ ]:
# Importação da função load_iris do módulo sklearn.datasets e do Pandas
from sklearn.datasets import load_iris
import pandas as pd

# Definição dos nomes das colunas que serão atribuídos ao DataFrame
iris_cols = ['Comprimento das Sépalas', 'Largura das Sépalas', 'Comprimento das Pétalas', 'Largura das Pétalas']

# Carregamento do conjunto de dados "iris"
iris_data = load_iris()

# Extração dos atributos previsores e dos rótulos de classe do conjunto de dados
X = iris_data.data  # Atributos previsores
y = iris_data.target  # Rótulos de classe

# Nomes das classes reais
y_names = iris_data.target_names

# Descrição do conjunto de dados
DESCR = iris_data.DESCR

# Criação de um DataFrame do Pandas com os atributos previsores
df = pd.DataFrame(X, columns=iris_cols)

# Geração de estatísticas descritivas para o DataFrame
df.describe()


In [ ]:
# Importação das bibliotecas necessárias
from sklearn.cluster import KMeans  # Para realizar o agrupamento K-means
import matplotlib.pyplot as pyp  # Para plotar gráficos
import pandas as pd  # Para trabalhar com DataFrames
from sklearn.metrics.cluster import silhouette_score, davies_bouldin_score, calinski_harabasz_score  # Métricas de avaliação de clusters

# Lista para armazenar os erros quadráticos relativos ao centróide
erros = []

# Lista para armazenar os índices de avaliação de cluster (silhueta, Davies-Bouldin e Calinski-Harabasz)
indices = [[0, 0, 0]]

# Range de valores de K (número de clusters) que serão testados
K_set = range(1, 11)

# Loop sobre os valores de K
for i in K_set:
    # Criação de um objeto KMeans com o número de clusters atual
    kmeans = KMeans(n_clusters=i, random_state=0)
    
    # Ajuste do modelo aos dados e previsão dos clusters para cada amostra
    prevs = kmeans.fit_predict(X)
    
    # Armazenamento do erro quadrático relativo ao centróide para este valor de K
    erros.append(kmeans.inertia_)

    # Calculando as métricas de avaliação do cluster apenas para K > 1
    if (i > 1):
        sil = silhouette_score(X, prevs, random_state=0)  # Silhueta
        dav = davies_bouldin_score(X, prevs)  # Davies-Bouldin
        cal = calinski_harabasz_score(X, prevs)  # Calinski-Harabasz
        
        # Armazenamento dos índices de avaliação
        indices = indices + [[sil, dav, cal]]

# Plotagem do gráfico de Elbow Method para ajudar a escolher o número ideal de clusters
pyp.plot(K_set, erros)
pyp.xlabel('K')  # Rótulo do eixo x
pyp.ylabel('Erro Quadrático relativo ao Centroide')  # Rótulo do eixo y
pyp.show()

# Criação de um DataFrame para exibir os índices de avaliação de cluster
df_indices = pd.DataFrame(data=indices, columns=['Silhueta', 'Davies-Bouldin', 'Calinski-Harabasz'], index=K_set)
print(df_indices)


In [ ]:
# Importação das bibliotecas necessárias
from sklearn.decomposition import PCA  # Para realizar a análise de componentes principais (PCA)
import seaborn as sb  # Para visualização de dados
pca = PCA(n_components=2, random_state=0)  # Criação de um objeto PCA com 2 componentes principais
Xpca = pca.fit_transform(X)  # Redução da dimensionalidade dos dados usando PCA
sb.pairplot(pd.DataFrame(Xpca, columns=['Atributo 1', 'Atributo 2']))  # Criação de um gráfico de pares para visualizar os dados reduzidos


In [ ]:
# Importação das bibliotecas necessárias
from sklearn.preprocessing import StandardScaler  # Para padronizar os dados
import numpy as np  # Para manipulação de arrays
Xps = StandardScaler().fit_transform(Xpca)  # Padronização dos dados reduzidos usando StandardScaler
XpcaXps = np.concatenate((Xpca, Xps), axis=1)  # Concatenação dos dados originais reduzidos com os dados padronizados
# Criação de um DataFrame para exibir as estatísticas descritivas dos dados originais e padronizados
df_XpcaXps = pd.DataFrame(XpcaXps, columns=['Original 1', 'Original 2', 'Balanceado 1', 'Balanceado 2'])
# Exibição das estatísticas descritivas dos dados
display(df_XpcaXps.describe())

In [ ]:
# Uso do KMeans para atribuir rótulos de cluster a cada amostra
from sklearn.cluster import KMeans  # Para realizar o agrupamento K-means
clusters = KMeans(n_clusters=3).fit_predict(Xps)

# Criação de um novo DataFrame com os dados originais e o rótulo do cluster atribuído a cada amostra
df_clusters = df.copy()
df_clusters['Espécie'] = clusters

# Criação de um gráfico de pares para visualizar a distribuição dos dados com base nos clusters atribuídos
sb.pairplot(df_clusters, hue='Espécie')

In [ ]:
# Exemplo 6 – Rotulação
# Rotula os grupos na mesma ordem em que aparecem para load_iris()
df_clusters['Espécie'] = df_clusters['Espécie'].map({0: 'setosa', 1: 'virginica', 2: 'versicolor'})
# Mapeia novamente os rótulos para os índices de acordo com load_iris()
y_clusters = df_clusters['Espécie'].map({'setosa': 0, 'versicolor': 1, 'virginica': 2}).values

In [ ]:
# Importação das bibliotecas necessárias
from sklearn import svm  # Para o modelo SVM
from sklearn.model_selection import train_test_split, GridSearchCV  # Para dividir os dados em conjuntos de treinamento e teste e realizar busca em grade
from sklearn.metrics import accuracy_score  # Para calcular a precisão do modelo

# Divisão dos dados em conjuntos de treinamento e teste
X_treino, X_teste, y_treino, y_teste_clusters = train_test_split(X, y_clusters, random_state=50)
_, _, _, y_teste_real = train_test_split(X, y, random_state=50)

# Definição dos valores a serem testados para os parâmetros C e gamma
Cs = [0.001, 0.01, 0.1, 1, 100, 1000]
gammas = [1e-2, 1e-3, 1e-4, 1e-5]

# Lista de dicionários contendo os parâmetros a serem testados para cada tipo de kernel
params_list = [
    {'kernel': ['linear'], 'C': Cs},
    {'kernel': ['poly'], 'C': Cs, 'gamma': gammas},
    {'kernel': ['rbf'], 'C': Cs, 'gamma': gammas}
]

# Configuração da busca em grade com cross-validation
grid = GridSearchCV(svm.SVC(probability=True, max_iter=1000000, random_state=0), params_list, scoring='accuracy')

# Ajuste do modelo aos dados de treinamento para encontrar os melhores parâmetros
grid.fit(X_treino, y_treino)

# Impressão dos melhores parâmetros encontrados pela busca em grade
print('Melhores parâmetros: {0}\n'.format(grid.best_params_))

# Extração das métricas de desempenho para cada conjunto de parâmetros testados
medias = grid.cv_results_['mean_test_score']
ranks = grid.cv_results_['rank_test_score']
desvios = grid.cv_results_['std_test_score']
params_set = grid.cv_results_['params']

# Organização e impressão das métricas de desempenho para cada conjunto de parâmetros testados
zipped = zip(ranks, medias, desvios, params_set)
sorted_zip = sorted(zipped, key=lambda x: x[0])
for rank, media, desvio, params in sorted_zip:
    print('Ranking: {0:0.4f} - Média: {1:0.04f} - Desvio: {2:0.04f} –Parâmetros {3}'.format(rank, media, desvio, params))


In [ ]:
# Criação do modelo SVM com os melhores parâmetros encontrados
model = svm.SVC(C=100, gamma=1e-2, kernel='rbf', max_iter=1000000, random_state=0)

# Treinamento do modelo SVM com os dados de treinamento
model.fit(X_treino, y_treino)

# Previsão dos rótulos de cluster para os dados de teste
y_svm_prevs = model.predict(X_teste)

# Medição da precisão do modelo SVM em diferentes cenários e exibição dos resultados
display(
    accuracy_score(y_teste_clusters, y_svm_prevs),  # Precisão entre os rótulos de cluster previstos e reais
    accuracy_score(y_teste_real, y_svm_prevs),      # Precisão entre os rótulos reais e os previstos pelo modelo SVM
    accuracy_score(y_teste_real, y_teste_clusters)  # Precisão entre os rótulos reais e os rótulos de cluster
)